In [0]:
payroll_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_payroll")
from pyspark.sql.functions import to_date, round, col, monotonically_increasing_id

payroll_df = payroll_df.withColumn("pay_period_start", to_date("pay_period_start", "dd-MM-yyyy HH:mm")) \
    .withColumn("pay_period_end", to_date("pay_period_end", "dd-MM-yyyy HH:mm")) \
    .withColumn("pay_date", to_date("pay_date", "dd-MM-yyyy HH:mm")) \
    .withColumn("total_compensation", (col("gross_salary") + col("bonus") + col("overtime_pay") + col("commission") + col("allowances")).cast("double")) \
    .withColumn("total_compensation", round(col("total_compensation"), 2)) \
    .withColumn("payroll_sk", monotonically_increasing_id()) \
    .withColumnRenamed("status", "payroll_status")

payroll_df.write.format("delta").mode("overwrite").saveAsTable("global_tech_sliver.transformed_data_silver.tf_payroll")

display(payroll_df)